In [1]:
import os
import itertools

import mlflow
import mlflow.sklearn
import pandas as pd
import sqlalchemy as sa
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("penguins-clasificacion")

engine = sa.create_engine(os.environ["DATA_DB_CONN"])
df = pd.read_sql("SELECT species, island, culmen_length_mm, culmen_depth_mm, "
                 "flipper_length_mm, body_mass_g, sex FROM clean.penguins_clean", engine)

print(f"Filas cargadas desde clean: {len(df)}")
print(f"MLflow: {mlflow.get_tracking_uri()}")
df.head()

2026/10/12 22:07:28 INFO mlflow.tracking.fluent: Experiment with name 'penguins-clasificacion' does not exist. Creating a new experiment.


Filas cargadas desde clean: 333
MLflow: http://mlflow:5000


,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
4,Adelie,Torgersen,39.3,20.6,190.0,3650.0,MALE


In [2]:
NUMERICAS = ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
CATEGORICAS = ["island", "sex"]
OBJETIVO = "species"

X = df[NUMERICAS + CATEGORICAS]
y = df[OBJETIVO]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Entrenamiento: {len(X_train)} | Prueba: {len(X_test)}")
print(f"\nDistribucion en entrenamiento:\n{y_train.value_counts()}")

Entrenamiento: 266 | Prueba: 67

Distribucion en entrenamiento:
species
Adelie       117
Gentoo        95
Chinstrap     54
Name: count, dtype: int64


In [3]:
def construir_pipeline(n_estimators, max_depth, min_samples_split):
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", StandardScaler(), NUMERICAS),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS),
        ])),
        ("clf", RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=42,
            n_jobs=-1,
        )),
    ])


malla = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 10],
    "min_samples_split": [2, 5, 10],
}

combinaciones = list(itertools.product(*malla.values()))
print(f"Total de ejecuciones a realizar: {len(combinaciones)}\n")

resultados = []

for i, (n_est, prof, min_split) in enumerate(combinaciones, start=1):
    with mlflow.start_run(run_name=f"rf_{i:02d}_n{n_est}_d{prof}_s{min_split}"):
        modelo = construir_pipeline(n_est, prof, min_split)
        modelo.fit(X_train, y_train)
        pred = modelo.predict(X_test)

        metricas = {
            "accuracy": accuracy_score(y_test, pred),
            "f1_macro": f1_score(y_test, pred, average="macro"),
            "precision_macro": precision_score(y_test, pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_test, pred, average="macro"),
        }

        mlflow.log_params({
            "n_estimators": n_est,
            "max_depth": prof,
            "min_samples_split": min_split,
            "modelo": "RandomForestClassifier",
            "filas_train": len(X_train),
            "filas_test": len(X_test),
        })
        mlflow.log_metrics(metricas)
        mlflow.sklearn.log_model(modelo, artifact_path="modelo",
                                 input_example=X_train.head(3))

        resultados.append({**dict(zip(malla.keys(), (n_est, prof, min_split))), **metricas})
        print(f"[{i:02d}/{len(combinaciones)}] n={n_est} d={prof} s={min_split} "
              f"-> acc={metricas['accuracy']:.4f} f1={metricas['f1_macro']:.4f}")

print("\nExperimentacion completada")

Total de ejecuciones a realizar: 27



2026/10/12 22:07:36 INFO mlflow.tracking._tracking_service.client: 🏃 View run rf_01_n50_d3_s2 at: http://mlflow:5000/#/experiments/1/runs/b204ef9e05bb4d0d962d99431d5d87c3.
2026/10/12 22:07:36 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


[01/27] n=50 d=3 s=2 -> acc=1.0000 f1=1.0000


2026/10/12 22:07:39 INFO mlflow.tracking._tracking_service.client: 🏃 View run rf_02_n50_d3_s5 at: http://mlflow:5000/#/experiments/1/runs/e933c08f62174713917dffe5560941b2.
2026/10/12 22:07:39 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


[02/27] n=50 d=3 s=5 -> acc=1.0000 f1=1.0000


...

[25/27] n=200 d=10 s=2 -> acc=1.0000 f1=1.0000


2026/10/12 22:08:53 INFO mlflow.tracking._tracking_service.client: 🏃 View run rf_26_n200_d10_s5 at: http://mlflow:5000/#/experiments/1/runs/e8d95058090044158df0c3940f4a1c20.
2026/10/12 22:08:53 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


[26/27] n=200 d=10 s=5 -> acc=1.0000 f1=1.0000


2026/10/12 22:08:57 INFO mlflow.tracking._tracking_service.client: 🏃 View run rf_27_n200_d10_s10 at: http://mlflow:5000/#/experiments/1/runs/b0db005580f7403f8f3a3c94a77518fa.
2026/10/12 22:08:57 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


[27/27] n=200 d=10 s=10 -> acc=1.0000 f1=1.0000

Experimentacion completada


In [4]:
mlflow.set_experiment("penguins-dos-variables")

NUMERICAS_2 = ["culmen_depth_mm", "body_mass_g"]
CATEGORICAS_2 = []

X2 = df[NUMERICAS_2]
y2 = df[OBJETIVO]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

def pipeline_2v(n_estimators, max_depth, min_samples_split):
    return Pipeline([
        ("prep", ColumnTransformer([("num", StandardScaler(), NUMERICAS_2)])),
        ("clf", RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=42,
            n_jobs=-1,
        )),
    ])

resultados_2v = []

for i, (n_est, prof, min_split) in enumerate(combinaciones, start=1):
    with mlflow.start_run(run_name=f"rf2v_{i:02d}_n{n_est}_d{prof}_s{min_split}"):
        modelo = pipeline_2v(n_est, prof, min_split)
        modelo.fit(X2_train, y2_train)
        pred = modelo.predict(X2_test)

        metricas = {
            "accuracy": accuracy_score(y2_test, pred),
            "f1_macro": f1_score(y2_test, pred, average="macro"),
            "precision_macro": precision_score(y2_test, pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y2_test, pred, average="macro"),
        }

        mlflow.log_params({
            "n_estimators": n_est,
            "max_depth": prof,
            "min_samples_split": min_split,
            "modelo": "RandomForestClassifier",
            "variables": ",".join(NUMERICAS_2),
            "n_variables": len(NUMERICAS_2),
        })
        mlflow.log_metrics(metricas)
        mlflow.sklearn.log_model(modelo, artifact_path="modelo",
                                 input_example=X2_train.head(3))

        resultados_2v.append({"n_estimators": n_est, "max_depth": prof,
                              "min_samples_split": min_split, **metricas})
        print(f"[{i:02d}/27] n={n_est} d={prof} s={min_split} "
              f"-> acc={metricas['accuracy']:.4f} f1={metricas['f1_macro']:.4f}")

print("\nSegunda experimentacion completada")

2026/10/12 22:08:57 INFO mlflow.tracking.fluent: Experiment with name 'penguins-dos-variables' does not exist. Creating a new experiment.


2026/10/12 22:09:00 INFO mlflow.tracking._tracking_service.client: 🏃 View run rf2v_01_n50_d3_s2 at: http://mlflow:5000/#/experiments/2/runs/ae4e547c3aca4671ba55c9d72f99a4d2.
2026/10/12 22:09:00 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/2.


[01/27] n=50 d=3 s=2 -> acc=0.7910 f1=0.6019


2026/10/12 22:09:03 INFO mlflow.tracking._tracking_service.client: 🏃 View run rf2v_02_n50_d3_s5 at: http://mlflow:5000/#/experiments/2/runs/188b941a43b047478ebc3312aa29adb0.
2026/10/12 22:09:03 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/2.


[02/27] n=50 d=3 s=5 -> acc=0.7910 f1=0.6019


...

[25/27] n=200 d=10 s=2 -> acc=0.7164 f1=0.6010


2026/10/12 22:10:16 INFO mlflow.tracking._tracking_service.client: 🏃 View run rf2v_26_n200_d10_s5 at: http://mlflow:5000/#/experiments/2/runs/0ec34333477e446fab64c5eadf78c444.
2026/10/12 22:10:16 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/2.


[26/27] n=200 d=10 s=5 -> acc=0.7463 f1=0.6172


2026/10/12 22:10:20 INFO mlflow.tracking._tracking_service.client: 🏃 View run rf2v_27_n200_d10_s10 at: http://mlflow:5000/#/experiments/2/runs/a01a1ed670fd4d00b075501d9e671b21.
2026/10/12 22:10:20 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/2.


[27/27] n=200 d=10 s=10 -> acc=0.7761 f1=0.6334

Segunda experimentacion completada


In [5]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
NOMBRE_MODELO = "penguins-classifier"

exp = client.get_experiment_by_name("penguins-clasificacion")

runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.accuracy DESC", "params.n_estimators ASC", "params.max_depth ASC"],
)

mejor = runs[0]
print(f"Run elegido: {mejor.data.tags.get('mlflow.runName')}")
print(f"  accuracy: {mejor.data.metrics['accuracy']:.4f}")
print(f"  f1_macro: {mejor.data.metrics['f1_macro']:.4f}")
print(f"  parametros: n_estimators={mejor.data.params['n_estimators']}, "
      f"max_depth={mejor.data.params['max_depth']}, "
      f"min_samples_split={mejor.data.params['min_samples_split']}")

uri = f"runs:/{mejor.info.run_id}/modelo"
version = mlflow.register_model(model_uri=uri, name=NOMBRE_MODELO)

print(f"\nModelo registrado: {NOMBRE_MODELO} version {version.version}")

Run elegido: rf_18_n100_d10_s10
  accuracy: 1.0000
  f1_macro: 1.0000
  parametros: n_estimators=100, max_depth=10, min_samples_split=10


Successfully registered model 'penguins-classifier'.
2026/10/12 22:10:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: penguins-classifier, version 1



Modelo registrado: penguins-classifier version 1


Created version '1' of model 'penguins-classifier'.


In [6]:
client.set_registered_model_alias(
    name=NOMBRE_MODELO,
    alias="produccion",
    version=version.version,
)

client.update_model_version(
    name=NOMBRE_MODELO,
    version=version.version,
    description=(
        "RandomForest sobre 6 variables (4 medidas + isla + sexo). "
        "Entrenado con 266 filas de clean.penguins_clean. "
        "Seleccionado por mayor accuracy y menor complejidad entre los empatados."
    ),
)

print(f"Alias 'produccion' asignado a la version {version.version}")
print(f"\nLa API cargara el modelo con: models:/{NOMBRE_MODELO}@produccion")

Alias 'produccion' asignado a la version 1

La API cargara el modelo con: models:/penguins-classifier@produccion


In [7]:
## Selección del modelo para el Model Registry

### Criterio de selección
'''Con seis variables, 26 de los 27 modelos alcanzaron exactitud perfecta sobre el
conjunto de prueba. Ante ese empate, se aplica el principio de parsimonia: a igual
desempeño, se prefiere el modelo más simple, porque generaliza mejor sobre datos
nuevos y resulta más barato de servir en inferencia. El criterio de desempate es,
en orden, menor n_estimators y menor max_depth.'''

### Versión 1: criterio mal implementado

'''El primer registro delegó el ordenamiento a `search_runs` mediante `order_by`:
    order_by=["metrics.accuracy DESC", "params.n_estimators ASC", "params.max_depth ASC"]

MLflow almacena los parámetros como cadenas de texto, de modo que el ordenamiento
fue lexicográfico y no numérico: `"100"` precede a `"50"` porque `"1"` < `"5"`, y
`"10"` precede a `"3"` por la misma razón. El resultado fue que se seleccionó
`rf_18_n100_d10_s10`, uno de los modelos más complejos del barrido, justo lo
contrario de lo que buscaba el criterio.'''

### Versión 2: criterio corregido

'''El segundo registro filtra primero los runs empatados en exactitud máxima y luego
los ordena en Python, convirtiendo los parámetros a entero antes de comparar:
    key=lambda r: (int(r.data.params["n_estimators"]), int(r.data.params["max_depth"]))

Con esto, la selección responde al criterio declarado.'''

### Por qué se conservan ambas versiones

'''No se elimina la versión 1. Mantener ambas en el Model Registry documenta la
corrección y deja constancia de qué modelo estuvo registrado en cada momento,
que es precisamente la trazabilidad que el registro de modelos busca aportar.
El alias `produccion` se reasigna a la versión 2, y es ese alias —no un
identificador de ejecución ni una ruta de almacenamiento— el que consume la API
de inferencia.'''

'No se elimina la versión 1. Mantener ambas en el Model Registry documenta la\ncorrección y deja constancia de qué modelo estuvo registrado en cada momento,\nque es precisamente la trazabilidad que el registro de modelos busca aportar.\nEl alias `produccion` se reasigna a la versión 2, y es ese alias —no un\nidentificador de ejecución ni una ruta de almacenamiento— el que consume la API\nde inferencia.'

In [8]:
runs_ok = [r for r in runs if r.data.metrics["accuracy"] == max(x.data.metrics["accuracy"] for x in runs)]

mejor = sorted(
    runs_ok,
    key=lambda r: (int(r.data.params["n_estimators"]), int(r.data.params["max_depth"])),
)[0]

print(f"Empatados en accuracy maxima: {len(runs_ok)}")
print(f"Mas simple entre ellos: {mejor.data.tags.get('mlflow.runName')}")
print(f"  n_estimators={mejor.data.params['n_estimators']}, "
      f"max_depth={mejor.data.params['max_depth']}, "
      f"min_samples_split={mejor.data.params['min_samples_split']}")

uri = f"runs:/{mejor.info.run_id}/modelo"
version = mlflow.register_model(model_uri=uri, name=NOMBRE_MODELO)
print(f"\nRegistrado como version {version.version}")

Registered model 'penguins-classifier' already exists. Creating a new version of this model...
2026/10/12 22:10:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: penguins-classifier, version 2


Empatados en accuracy maxima: 26
Mas simple entre ellos: rf_03_n50_d3_s10
  n_estimators=50, max_depth=3, min_samples_split=10

Registrado como version 2


Created version '2' of model 'penguins-classifier'.


In [9]:
client.set_registered_model_alias(
    name=NOMBRE_MODELO,
    alias="produccion",
    version=version.version,
)

client.update_model_version(
    name=NOMBRE_MODELO,
    version=version.version,
    description=(
        "RandomForest sobre 6 variables (4 medidas + isla + sexo). "
        "n_estimators=50, max_depth=3, min_samples_split=10. "
        "Entrenado con 266 filas de clean.penguins_clean. "
        "Seleccionado por mayor accuracy y menor complejidad entre los 26 empatados."
    ),
)

print(f"Alias 'produccion' -> version {version.version}")
print(f"La API cargara: models:/{NOMBRE_MODELO}@produccion")

Alias 'produccion' -> version 2
La API cargara: models:/penguins-classifier@produccion
